# B6 (Partie 3) — Améliorations de l'Auto-encodeur

**Prérequis** : TP3 exécuté (constat : AUROC = 0.465, FN = 53/63).

Ce TP implémente les trois pistes d'amélioration identifiées :
1. **Goulot resserré** — ratio ≥ 4 pour éliminer la quasi-identité
2. **Perte MSE+SSIM** — plus sensible aux altérations de texture
3. **PatchCore-lite** — backbone pré-entraîné ResNet50, sans entraînement from scratch

## §0 — Setup

In [ ]:
import os, sys
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("src")))

from indusense.vision.dataset import load_train_val, load_test, load_masks
from indusense.vision.augment import build_pipeline
from indusense.vision.model   import build_autoencoder, compression_ratio, mse_ssim_loss
from indusense.vision.train   import train as ae_train
from indusense.vision.anomaly import (
    reconstruction_errors, calibrate_threshold,
    pixel_error_map, plot_heatmap,
    plot_score_histogram, evaluate, plot_confusion_matrix,
)

BOTTLE_ROOT = Path("bottle")
SEED        = 42
Path("figures").mkdir(exist_ok=True)
Path("checkpoints").mkdir(exist_ok=True)

X_train, X_val = load_train_val(BOTTLE_ROOT, val_ratio=0.2, seed=SEED)
X_test, y_test, test_classes = load_test(BOTTLE_ROOT)
masks = load_masks(BOTTLE_ROOT)

defect_classes = sorted(set(c for c in test_classes if c != "good"))
test_arr = np.array(test_classes)
print(f"Train={X_train.shape[0]}  Val={X_val.shape[0]}  Test={X_test.shape[0]}")

# Baseline TP3 à battre
BASELINE = {"label": "TP3 baseline", "auroc": 0.465, "recall": 0.159, "fn": 53}

## §1 — Amélioration 1 : Goulot resserré (ratio ≈ 12)

On ajoute une 4e couche Conv2D dans l'encodeur : `filters=(32, 64, 128, 64)`.

- Goulot : 16×16×64 = **16 384** valeurs
- Ratio  : 196 608 / 16 384 ≈ **12.0** (vs 1.5 en TP3)

Le modèle est forcé à apprendre une représentation compacte → il ne peut plus copier les défauts.

In [ ]:
model_v2 = build_autoencoder(input_shape=(256, 256, 3), filters=(32, 64, 128, 64))
model_v2.summary()

ratio_v2 = compression_ratio(model_v2)
print(f"\nRatio de compression : {ratio_v2:.1f}  ({'✓ compact' if ratio_v2 >= 4 else '⚠ trop faible'})")

In [ ]:
history_v2 = ae_train(
    model_v2, X_train, X_val,
    pipeline=build_pipeline(),
    epochs=50, batch_size=16, learning_rate=1e-3,
    loss="mse",
    run_name="ae-bottle-ratio12-mse",
    checkpoint_path="checkpoints/ae_v2_best.keras",
)

# ── Évaluation ────────────────────────────────────────────────────────────────
errors_val_v2  = reconstruction_errors(model_v2, X_val)
errors_test_v2 = reconstruction_errors(model_v2, X_test)
threshold_v2   = calibrate_threshold(errors_val_v2, method="percentile", percentile=99)
metrics_v2     = evaluate(y_test, errors_test_v2, threshold_v2)

print(f"\n=== §1 — Goulot resserré (ratio ≈ 12, MSE) ===")
print(f"  AUROC   : {metrics_v2['auroc']:.3f}  (baseline: {BASELINE['auroc']:.3f})")
print(f"  Rappel  : {metrics_v2['recall']:.2%}  (baseline: {BASELINE['recall']:.2%})")
print(f"  FN      : {metrics_v2['fn']}  (baseline: {BASELINE['fn']})")

plot_score_histogram(
    errors_val_v2,
    errors_test_v2[test_arr == "good"],
    errors_test_v2[test_arr != "good"],
    threshold=threshold_v2,
    save_path="figures/tp4_01_histogram_v2.png",
)
plot_confusion_matrix(metrics_v2, save_path="figures/tp4_01_confusion_v2.png")

## §2 — Amélioration 2 : Perte MSE+SSIM

Même architecture (ratio ≈ 12), mais perte `0.8 × MSE + 0.2 × (1 − SSIM)`.  
SSIM compare la structure locale (contraste, luminance, corrélation) plutôt que les valeurs pixel absolues — plus adapté aux défauts de texture.

In [ ]:
model_v3 = build_autoencoder(input_shape=(256, 256, 3), filters=(32, 64, 128, 64))
loss_ssim = mse_ssim_loss(alpha=0.8)

history_v3 = ae_train(
    model_v3, X_train, X_val,
    pipeline=build_pipeline(),
    epochs=50, batch_size=16, learning_rate=1e-3,
    loss=loss_ssim,
    run_name="ae-bottle-ratio12-mse-ssim",
    checkpoint_path="checkpoints/ae_v3_best.keras",
)

# ── Évaluation ────────────────────────────────────────────────────────────────
errors_val_v3  = reconstruction_errors(model_v3, X_val)
errors_test_v3 = reconstruction_errors(model_v3, X_test)
threshold_v3   = calibrate_threshold(errors_val_v3, method="percentile", percentile=99)
metrics_v3     = evaluate(y_test, errors_test_v3, threshold_v3)

print(f"\n=== §2 — MSE+SSIM (ratio ≈ 12) ===")
print(f"  AUROC   : {metrics_v3['auroc']:.3f}")
print(f"  Rappel  : {metrics_v3['recall']:.2%}")
print(f"  FN      : {metrics_v3['fn']}")

plot_score_histogram(
    errors_val_v3,
    errors_test_v3[test_arr == "good"],
    errors_test_v3[test_arr != "good"],
    threshold=threshold_v3,
    save_path="figures/tp4_02_histogram_v3.png",
)
plot_confusion_matrix(metrics_v3, save_path="figures/tp4_02_confusion_v3.png")

## §3 — PatchCore-lite (backbone ResNet50 pré-entraîné)

**Principe PatchCore** :
1. Extraire des features de **patches** (sous-régions) de chaque image saine via un backbone pré-entraîné (ResNet50, ImageNet).
2. Constituer une **banque mémoire** de features saines (coreset).
3. En inférence : calculer la **distance min** entre les features d'une image et la banque mémoire.
   → Distance élevée = image non conforme = anomalie.

Aucun entraînement from scratch — le backbone a déjà appris à représenter les structures visuelles.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from sklearn.neighbors import NearestNeighbors

# ── 1. Backbone ResNet50 pré-entraîné ImageNet ────────────────────────────────
# Couche intermédiaire : conv3_block4_out → (N, 32, 32, 512) pour 256×256
base = keras.applications.ResNet50(include_top=False, input_shape=(256, 256, 3),
                                    weights="imagenet")
feat_model = keras.Model(base.input, base.get_layer("conv3_block4_out").output)
feat_model.trainable = False

print(f"Feature shape (1 image) : {feat_model.output_shape[1:]}")
print(f"Patches par image       : {feat_model.output_shape[1] * feat_model.output_shape[2]}")

In [ ]:
# ── 2. Banque mémoire — features des images saines d'entraînement ─────────────
print("Extraction des features train...")
feats_train = feat_model.predict(X_train, batch_size=8, verbose=1)  # (168, 32, 32, 512)
F = feats_train.shape[-1]
patches_train = feats_train.reshape(-1, F)                           # (168×1024, 512)

# Coreset : sous-échantillonnage aléatoire (10%) pour limiter la mémoire
rng = np.random.default_rng(SEED)
n_coreset = max(1000, len(patches_train) // 10)
idx_core  = rng.choice(len(patches_train), size=n_coreset, replace=False)
memory_bank = patches_train[idx_core]
print(f"Banque mémoire : {memory_bank.shape[0]} patches × {F} features")

# ── 3. kNN — 1 voisin dans la banque mémoire ─────────────────────────────────
nn = NearestNeighbors(n_neighbors=1, algorithm="ball_tree", n_jobs=-1)
nn.fit(memory_bank)
print("kNN ajusté sur la banque mémoire.")

In [ ]:
# ── 4. Score d'anomalie = distance max sur tous les patches de l'image ─────────
print("Calcul des scores PatchCore sur le test set...")
scores_pc = []
for img in X_test:
    f = feat_model.predict(img[np.newaxis], verbose=0)  # (1, 32, 32, 512)
    p = f.reshape(-1, F)                                # (1024, 512)
    dists, _ = nn.kneighbors(p)
    scores_pc.append(float(dists.max()))
scores_pc = np.array(scores_pc)

# Score de référence sur val (pour le seuil)
scores_val_pc = []
for img in X_val:
    f = feat_model.predict(img[np.newaxis], verbose=0)
    p = f.reshape(-1, F)
    dists, _ = nn.kneighbors(p)
    scores_val_pc.append(float(dists.max()))
scores_val_pc = np.array(scores_val_pc)

threshold_pc = calibrate_threshold(scores_val_pc, method="percentile", percentile=99)
metrics_pc   = evaluate(y_test, scores_pc, threshold_pc)

print(f"\n=== §3 — PatchCore-lite (ResNet50 backbone) ===")
print(f"  AUROC   : {metrics_pc['auroc']:.3f}")
print(f"  Rappel  : {metrics_pc['recall']:.2%}")
print(f"  FN      : {metrics_pc['fn']}")

plot_confusion_matrix(metrics_pc, save_path="figures/tp4_03_confusion_pc.png")

## §4 — Comparaison synthétique

In [ ]:
rows = [
    ("TP3 baseline",           "ratio≈1.5 / MSE",    BASELINE["auroc"],  BASELINE["recall"], BASELINE["fn"]),
    ("§1 Goulot resserré",     "ratio≈12  / MSE",    metrics_v2["auroc"], metrics_v2["recall"], metrics_v2["fn"]),
    ("§2 MSE+SSIM",            "ratio≈12  / MSE+SSIM", metrics_v3["auroc"], metrics_v3["recall"], metrics_v3["fn"]),
    ("§3 PatchCore-lite",      "ResNet50 backbone",  metrics_pc["auroc"], metrics_pc["recall"], metrics_pc["fn"]),
]

print(f"{'Modèle':<24} {'Configuration':<24} {'AUROC':>6} {'Rappel':>8} {'FN':>4}")
print("-" * 70)
for label, config, auroc, recall, fn in rows:
    marker = " ← baseline" if label == "TP3 baseline" else ""
    print(f"{label:<24} {config:<24} {auroc:>6.3f} {recall:>7.1%} {fn:>4}{marker}")

## §5 — Analyse comparative

### Goulot resserré (§1)
Augmenter le ratio de compression force le modèle à encoder l'essentiel des images saines, sans pouvoir mémoriser les détails fins (dont les défauts). L'AUROC devrait augmenter significativement par rapport au baseline.

### MSE+SSIM (§2)
La perte SSIM pénalise les différences structurelles locales plutôt que les valeurs absolues de pixel. Elle est plus sensible aux défauts de texture (`broken_small`, `contamination`) que MSE seul.

### PatchCore-lite (§3)
L'approche feature-based évite le problème d'identité par construction : le backbone (ResNet50, ImageNet) n'a **pas été entraîné à reconstruire des bouteilles** — il extrait des descripteurs génériques. La distance dans l'espace feature est un indicateur d'anomalie bien plus robuste que l'erreur de reconstruction pixel à pixel.

### Conclusion
L'auto-encodeur de reconstruction est un bon point de départ **pédagogique** mais atteint ses limites rapidement. Les méthodes basées sur des features pré-entraînées (PatchCore, PaDiM) résolvent structurellement le problème d'identité et dominent aujourd'hui sur MVTec AD.